## Exercise 3: Two Continuous Stirred Tank Reactors (CSTRs) in Series

### Problem Statement
We consider a biological system consisting of two continuous stirred tank reactors (CSTRs) operating in series under steady-state conditions. The goal is to predict the outlet concentrations of cell mass ($X$) and substrate ($S$) for various volume configurations and compare them to a single 1000 L reactor.

**Given Parameters:**
* Flow rate: $F = 100 \text{ L/h}$
* Inlet substrate concentration: $S_0 = 10 \text{ g/L}$
* Yield coefficient: $Y_{X/S} = 0.5 \text{ g cells/g substrate}$
* Maximum growth rate: $\mu_{\max} = 1 \text{ h}^{-1}$
* Monod constant: $K_S = 0.75 \text{ g/L}$

**Assumptions:**
1. Steady-state operation.
2. Maintenance energy of the cells is negligible.
3. The specific growth rate follows Monod kinetics: $\mu(S) = \mu_{\max} \frac{S}{K_S + S}$
4. The feed entering the first reactor is sterile ($X_0 = 0$).

### Mathematical Model

The system is modeled using mass balances for both biomass and substrate across both reactors. The general balance equation is: **Accumulation = In - Out + Generation/Consumption**. Under steady-state conditions, Accumulation = 0.

#### Reactor 1 Balances
Reactor 1 receives a sterile feed.
* **Biomass:**
    $$0 = - D_1 X_1 + \mu(S_1) X_1$$
* **Substrate:**
    $$0 = D_1 (S_0 - S_1) - \frac{1}{Y_{X/S}} \mu(S_1) X_1$$

#### Reactor 2 Balances
Reactor 2 is fed directly by the effluent of Reactor 1. Therefore, the biomass balance must account for the cells entering the reactor ($D_2 X_1$).
* **Biomass:**
    $$0 = D_2 X_1 - D_2 X_2 + \mu(S_2) X_2$$
* **Substrate:**
    $$0 = D_2 (S_1 - S_2) - \frac{1}{Y_{X/S}} \mu(S_2) X_2$$

Where the dilution rate is defined as $D_i = \frac{F}{V_i}$.

In [ ]:
import numpy as np
from scipy.optimize import fsolve

# Define Parameters 
F = 100      # L/h
S0 = 10      # g/L
Yxs = 0.5    # g/g
mu_max = 1.0 # h^-1
Ks = 0.75    # g/L

# Define the Non-Linear System
def system(vars, V1, V2):
    S1, X1, S2, X2 = vars
    
    # Dilution rates
    D1 = F / V1
    D2 = F / V2
    
    # Monod kinetics
    mu1 = mu_max * S1 / (Ks + S1)
    mu2 = mu_max * S2 / (Ks + S2)
    
    # Reactor 1 Balances (sterile feed)
    eq1 = -D1 * X1 + mu1 * X1 
    eq2 = D1 * (S0 - S1) - (1 / Yxs) * mu1 * X1
    
    # Reactor 2 Balances (receives X1 and S1 from Reactor 1)
    eq3 = D2 * X1 - D2 * X2 + mu2 * X2
    eq4 = D2 * (S1 - S2) - (1 / Yxs) * mu2 * X2
    
    return [eq1, eq2, eq3, eq4]

def solve_case(V1, V2):
    # Initial guess assumes active biomass. If a reactor washes out, 
    # the solver will naturally drive X down to 0 and S up to 10.
    initial_guess = [0.5, 4.5, 0.5, 4.5] 
    
    # full_output=True allows us to monitor solver convergence
    sol, infodict, ier, mesg = fsolve(system, initial_guess, args=(V1, V2), full_output=True)
    
    if ier != 1:
        print(f"  [!] Warning: Solver struggled to converge for V1={V1}, V2={V2}.")
        
    return sol

### Simulation and Results

We will now solve the non-linear system for the four proposed volume configurations, as well as a baseline single 1000 L reactor, and compare the final outlet concentrations.

In [2]:
# 1. Run the Single Reactor Baseline
def single_reactor(vars):
    S, X = vars
    D = F / 1000
    mu = mu_max * S / (Ks + S)
    
    eq1 = -D * X + mu * X
    eq2 = D * (S0 - S) - (1 / Yxs) * mu * X
    return [eq1, eq2]

S_sing, X_sing = fsolve(single_reactor, [0.5, 4.5])
print("=== Single Reactor Result (1000 L) ===")
print(f"  Outlet: S = {S_sing:.3f} g/L, X = {X_sing:.3f} g/L\n")

# 2. Run the Two-Stage Configurations
cases = {
    "a": (800, 200),
    "b": (200, 800),
    "c": (900, 100),
    "d": (100, 900),
}

print("=== Two-Stage Reactor Configurations ===")
for key, (V1, V2) in cases.items():
    S1, X1, S2, X2 = solve_case(V1, V2)
    
    # Clean up near-zero floating point artifacts (e.g., -1.2e-15 -> 0.0)
    S1, X1 = max(0.0, S1), max(0.0, X1)
    S2, X2 = max(0.0, S2), max(0.0, X2)

    print(f"Case {key}) V1 = {V1} L, V2 = {V2} L:")
    print(f"  Reactor 1: S1 = {S1:.3f} g/L, X1 = {X1:.3f} g/L")
    print(f"  Reactor 2: S2 = {S2:.3f} g/L, X2 = {X2:.3f} g/L\n")

=== Single Reactor Result (1000 L) ===
  Outlet: S = 0.083 g/L, X = 4.958 g/L

=== Two-Stage Reactor Configurations ===
Case a) V1 = 800 L, V2 = 200 L:
  Reactor 1: S1 = 0.107 g/L, X1 = 4.946 g/L
  Reactor 2: S2 = 0.004 g/L, X2 = 4.998 g/L

Case b) V1 = 200 L, V2 = 800 L:
  Reactor 1: S1 = 0.750 g/L, X1 = 4.625 g/L
  Reactor 2: S2 = 0.007 g/L, X2 = 4.996 g/L

Case c) V1 = 900 L, V2 = 100 L:
  Reactor 1: S1 = 0.094 g/L, X1 = 4.953 g/L
  Reactor 2: S2 = 0.007 g/L, X2 = 4.997 g/L

Case d) V1 = 100 L, V2 = 900 L:
  Reactor 1: S1 = 10.000 g/L, X1 = 0.000 g/L
  Reactor 2: S2 = 0.094 g/L, X2 = 4.953 g/L



### Analysis and Comparison

**Single Reactor Baseline (1000 L):**
* **Final Output:** $S = 0.083 \text{ g/L}$, $X = 4.958 \text{ g/L}$
* In a single CSTR, the specific growth rate must exactly equal the dilution rate ($\mu = D$). To achieve a low substrate concentration, the dilution rate must be kept low, which limits the overall throughput.

**Two-Stage Configuration Analysis:**
The data reveals that dividing the total 1000 L volume into two active stages (Cases A, B, and C) drastically improves the efficiency of the system compared to a single reactor. 

* Cases A, B, and C: In all three of these configurations, the final substrate concentration drops to near-zero levels ($0.004 \text{ to } 0.007 \text{ g/L}$), heavily outperforming the single 1000 L reactor ($0.083 \text{ g/L}$). 
    * *Why this happens:* In Reactor 2, the biomass is constantly replenished by the inflow from Reactor 1. The mathematical consequence is that Reactor 2 is no longer bound by the $\mu = D$ constraint. Because it doesn't need to grow fast enough to overcome washout, $\mu_2$ can drop to near zero. A tiny $\mu_2$ correlates to a tiny $S_2$, allowing Reactor 2 to act as a highly efficient "polishing" stage that scrubs the remaining substrate from the liquid.
    * Case A ($S_2 = 0.004 \text{ g/L}$) is technically the most efficient at consuming substrate, though Cases B and C are practically identical in performance.

* **Case D (V1 = 100 L, V2 = 900 L) - The Washout Condition:**
    * **Final Output:** $S_2 = 0.094 \text{ g/L}$, $X_2 = 4.953 \text{ g/L}$
    * In this configuration, $V_1 = 100 \text{ L}$ and $F = 100 \text{ L/h}$, meaning the dilution rate $D_1 = 1.0 \text{ h}^{-1}$. Because $\mu_{\max}$ is also $1.0 \text{ h}^{-1}$, the specific growth rate can never exceed the dilution rate. Consequently, **Reactor 1 completely washes out** ($X_1 = 0 \text{ g/L}$, $S_1 = 10 \text{ g/L}$). 
    * Because Reactor 1 fails to cultivate any biomass, it acts merely as a pipe feeding raw substrate into Reactor 2. Reactor 2 is forced to act as a standalone 900 L reactor, which explains why its final output ($S = 0.094 \text{ g/L}$) is slightly worse than the baseline 1000 L reactor.

**Conclusion:**
This simulation illustrates why fermentations with separated growth and product-formation steps utilize multiple reactors in series. By using a first reactor to cultivate a dense biomass population, the second reactor is freed from the strict $\mu = D$ growth constraints. This allows the second reactor to operate at extreme efficiencies, driving substrate levels down to nearly zero and maximizing the final cell yield.